# Movement Type Model Deployment

---------

## Intro

This notebook automates the deployment of an AI model built with [DEEPCRAFT™ Studio](https://developer.imagimob.com/deepcraft-studio) onto the [PSOC™ Edge AI Kit (KIT_PSE84_AI)](https://www.infineon.com/evaluation-board/KIT-PSE84-AI) Infineon™ MCU development board using a ModusToolbox™ code example application.

More specifically, this notebook demonstrates how to deploy the [Movement Type Detection](https://github.com/Infineon/deepcraft-studio-accelerators/tree/main/MovementTypeDetection) model trained using the corresponding [DEEPCRAFT™ Studio Accelerator](https://github.com/Infineon/deepcraft-studio-accelerators/tree/main) project. This model uses IMU sensor data to classify whether the user is shaking the board or turning it in circles.

For the deployment of this specific project, you will use the [PSOC™ Edge Machine Learning DEEPCRAFT™ Deploy Motion](https://github.com/Infineon/mtb-example-psoc-edge-ml-deepcraft-deploy-motion) code example, which is already set up (1) to stream data from the IMU sensor of the PSOC™ Edge AI Kit into the model and (2) to run model inference on the Ethos-U55 neural network accelerator of the PSOC™ Edge AI Kit.

By running this notebook, you will:

1. **Parse** the `.h5` DEEPCRAFT™ Studio model in the `models/` folder to inspect its preprocessor, neural network architecture, and expected performance metrics
2. **Embed** the int8x8 quantized DEEPCRAFT™ edge code (`model.c` / `model.h`) from the `fw/` folder into the ModusToolbox™ application firmware
3. **Build & program** the board by running `make clean`, `make build`, and `make program` directly from this notebook

The deployment steps illustrated in this notebook can be applied to any other DEEPCRAFT™ Studio model or DEEPCRAFT™ Studio Accelerator as long as (1) you have the (quantized) DEEPCRAFT™ edge code (`model.c` / `model.h`) and (2) you can use one of the existing ModusToolbox™ code example applications.

---------

## Table of Contents

| Section | Description |
|--------|-------------|
| **1. Project Setup** | Import libraries, create ModusToolbox™ application, and configure project path and Modus Shell (e.g. `Cygwin.bat`). |
| **2. (Optional) Parse Model** | Inspect the H5 model in `models/`: classes, preprocessor, metrics, and Keras architecture. |
| **3. Model Deployment** | Copy `fw/` into the application `model/` folder, then run **make clean**, **make build**, and **make program**. |
| **4. Model Live Testing** | Read model predictions over UART: open a terminal app (Tera Term / PuTTY / screen) or launch a Python serial reader. |

---

## Prerequisites

- Install **ModusToolbox™ Tools** using the default settings (see the [README](README.md) for details).
- Set up the Python environment using `src/requirements.txt` (see the [README](README.md) for details).

------------------------------------------------------

## 1. Project Setup

### STEP 1. Import relevant libraries and set project paths

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TF INFO/WARNING (e.g. oneDNN messages)

import shutil
import subprocess
import sys
from pathlib import Path

# All paths are relative to the deploy root (the folder containing this notebook)
DEPLOY_ROOT = Path.cwd()
SRC_DIR = DEPLOY_ROOT / "src"
MODELS_DIR = DEPLOY_ROOT / "models"
BINARY_DIR = DEPLOY_ROOT / "psoc_edge_fw_binary"

# Make local helper modules importable (studio_h5_parser, modus_make)
sys.path.insert(0, str(SRC_DIR))

from studio_h5_parser import find_h5_file, parse_h5_model
from modus_make import run_make, open_programmer, flash_hex
from serial_monitor import check_board_connection

print(f"Deploy root:   {DEPLOY_ROOT}")
print(f"Models:        {MODELS_DIR}")
print(f"Binary folder: {BINARY_DIR}")

### STEP 2. Set ModusToolbox™ Tools and Shell paths

In [ ]:
# Override with a custom path if ModusToolbox is not installed in the default location.
# Example: Path(r"C:\Tools\ModusToolbox")
MANUAL_MODUSTOOLBOX_BASE = None

# Default: ~/ModusToolbox (standard install location on Windows)
search_base = MANUAL_MODUSTOOLBOX_BASE or (Path.home() / "ModusToolbox")
if not search_base.is_dir():
    raise FileNotFoundError(
        f"ModusToolbox base folder not found: {search_base}\n"
        "Set MANUAL_MODUSTOOLBOX_BASE to your install location."
    )

# Find all tools_* directories that contain a working Modus Shell (Cygwin.bat)
tool_candidates = [
    p for p in search_base.glob("tools_*")
    if p.is_dir() and (p / "modus-shell" / "Cygwin.bat").is_file()
]
if not tool_candidates:
    raise FileNotFoundError(
        f"No valid tools_* folder found under: {search_base}\n"
        "Set MANUAL_MODUSTOOLBOX_BASE to the folder that contains tools_<version>."
    )

# Use the latest installed version (e.g. tools_3.6 over tools_3.2)
MODUS_TOOLS_PATH = max(
    tool_candidates,
    key=lambda p: tuple(int(x) for x in p.name.split("_", 1)[1].split(".")),
)

# Cygwin.bat sets up PATH/CYSDK so that make commands run in the correct environment
MODUS_SHELL_PATH = MODUS_TOOLS_PATH / "modus-shell" / "Cygwin.bat"
print("Using MODUS_TOOLS_PATH:", MODUS_TOOLS_PATH)
print("Using MODUS_SHELL_PATH:", MODUS_SHELL_PATH)

### STEP 3. Create ModusToolbox™ Motion Application (First Time Only / To Create New Project)

To deploy the model using ModusToolbox™, you first need to use ModusToolbox™ Project Creator to create the application that will build the firmware hosting the model. Open Project Creator by running the cell below:


In [ ]:
subprocess.Popen([MODUS_TOOLS_PATH / "project-creator" / "project-creator.exe"])

Once the Project Creator window opens, follow the steps below to create your application:

1. In the `Kit Name` column, expand `PSOC Edge BSPs`
2. Select `KIT_PSE84_AI` as the kit and click `Next`
3. In `Application(s) Root Path`, click `Browse...` and select the folder that will contain your application. We recommend creating a dedicated folder for it
4. Select `None/Command line` as `Target IDE`. Select `Microsoft Visual Studio Code` if you plan to use Visual Studio Code to customize and debug your application
5. In the `Template Application` column, expand `Machine Learning`
6. Select `PSOC Edge Machine Learning DEEPCRAFT Deploy Motion`, modify the application folder name on the right side if needed, and click `Create`
7. Wait for the project creation process to complete successfully
8. Once completed, copy the path shown in `Application(s) Root Path` — you will need it in the next step
9. Click `Close` to close Project Creator and return to this notebook

### STEP 4. Set ModusToolbox™ Motion Project Path

Set the `MTB_PROJECT_PATH_ROOT` variable below to the `Application(s) Root Path` copied in the previous step.

In [ ]:
# Set this to the Application(s) Root Path shown in ModusToolbox Project Creator
MTB_PROJECT_PATH_ROOT = Path(r"C:\path\to\your\MTB\project")

The application folder inside it will be detected automatically when running the code below.

In [ ]:
# Auto-detect the application folder (most recently created, ignoring mtb_shared)
EXCLUDED_DIRS = {".metadata", "mtb_shared"}
project_dirs = [
    d for d in MTB_PROJECT_PATH_ROOT.iterdir()
    if d.is_dir() and d.name not in EXCLUDED_DIRS
]
if not project_dirs:
    raise FileNotFoundError(
        f"No project folders found in {MTB_PROJECT_PATH_ROOT} "
        f"(excluded: {EXCLUDED_DIRS})"
    )
MTB_PROJECT_NAME = max(project_dirs, key=lambda d: d.stat().st_ctime).name

MTB_PROJECT_PATH = MTB_PROJECT_PATH_ROOT / MTB_PROJECT_NAME
print(f"Project name (auto-detected): {MTB_PROJECT_NAME}")
print(f"Project path: {MTB_PROJECT_PATH}")

# Destination for the edge code files inside the CM55 core sub-project
MODEL_DEST = MTB_PROJECT_PATH / "proj_cm55" / "model"

---------------------------------------------------------

## 2. (Optional) Parse Model

Find the H5 model and run the parser to inspect classes, preprocessor, metrics, and architecture.

In [ ]:
h5_path = find_h5_file(MODELS_DIR)
if h5_path:
    parse_h5_model(h5_path)
else:
    print("No .h5 file found in models/. Add a model and re-run.")

--------------------------------------------------------------------------

## 3. Model Deployment

### STEP 1. Copy model to ModusToolbox™ application

Copy the edge code files from `fw/` to the `model` folder inside the `proj_cm55` sub-project of the ModusToolbox™ application.

In [ ]:
FW_DIR = DEPLOY_ROOT / "fw"
MODEL_DEST.mkdir(parents=True, exist_ok=True)
for f in FW_DIR.iterdir():
    if f.is_file():
        shutil.copy2(f, MODEL_DEST / f.name)
        print(f"Copied {f.name} -> {MODEL_DEST / f.name}")

### STEP 2. Clean ModusToolbox™ project

**Make clean** – Run this to clean the ModusToolbox™ build.

In [ ]:
run_make("clean", MODUS_SHELL_PATH, MTB_PROJECT_PATH)

### STEP 3. Build ModusToolbox™ project and save binary file

**Make build** – Run this to build the project and save the binary .hex file so it can be flashed later without rebuilding.
**NOTE** Building can take up to 4–5 minutes.

In [ ]:
run_make("build", MODUS_SHELL_PATH, MTB_PROJECT_PATH)

# Save a copy of the built binary so it can be flashed later without rebuilding
src_hex = MTB_PROJECT_PATH / "build" / "app_combined.hex"
if not src_hex.is_file():
    raise FileNotFoundError(f"Build output not found: {src_hex}")

BINARY_DIR.mkdir(parents=True, exist_ok=True)
dst_hex = BINARY_DIR / "app_combined.hex"
shutil.copy2(src_hex, dst_hex)
print(f"Copied HEX to: {dst_hex}")

### STEP 4. Program your board

**Connect the board** — Before programming, plug the PSOC™ Edge AI Kit into your PC using the `KitProg3` USB-C port on the board. Run the cell below to verify the board is detected.

Then choose one of the options below:

- **Option A** — `make program` (uses the ModusToolbox™ project build output)
- **Option B** — Open **ModusToolbox™ Programmer** to flash the `.hex` file saved in `psoc_edge_fw_binary/`
- **Option C** — Flash the `.hex` file via **OpenOCD** from the command line (experimental — may not work reliably on PSOC™ Edge)

In [ ]:
check_board_connection()

#### Option A — Make program

In [ ]:
run_make("program", MODUS_SHELL_PATH, MTB_PROJECT_PATH)

#### Option B — Flash .hex file with ModusToolbox™ Programmer

Opens [ModusToolbox™ Programmer](https://softwaretools.infineon.com/tools/com.ifx.tb.tool.modustoolboxprogtools) so you can flash the `.hex` file from `psoc_edge_fw_binary/` onto the board.

In [ ]:
open_programmer(BINARY_DIR / "app_combined.hex")

Once ModusToolbox™ Programmer opens, follow these steps:

1. Click **Open** and find and select the hex file `psoc_edge_fw_binary/app_combined.hex` saved after building the application at STEP 3
2. Check the **External Memory** box — this is required because the combined hex includes application code stored in the external QSPI/SMIF flash (in addition to the internal RRAM)
3. Select **KitProg3** as Programmer, then **KIT_PSE84_AI** as Board and click on **Connect**
4. Click **Program** to flash the board

#### Option C — Flash .hex file with OpenOCD (experimental)

Flash the `.hex` file from `psoc_edge_fw_binary/` using standalone [OpenOCD](https://softwaretools.infineon.com/tools/com.ifx.tb.tool.modustoolboxprogtools) (included in ModusToolbox™ Programming Tools).

The combined hex contains data for both the internal RRAM and the external QSPI/SMIF flash (where the CM33 and CM55 application code is stored). OpenOCD needs the BSP-generated `qspi_config.cfg` to know about the external flash banks; the function locates it automatically from the ModusToolbox™ project path set in Step 4 of Section 1.

> **Note:** This approach is experimental. OpenOCD may report a successful flash but fail to start the firmware on PSOC™ Edge due to its multi-core secure boot architecture. If the board does not respond after flashing, use **Option A** or **Option B** instead.

In [ ]:
flash_hex(BINARY_DIR / "app_combined.hex", mtb_project_path=MTB_PROJECT_PATH)

---------

## 4. Model Live Testing

Once the board is programmed, the firmware streams model predictions over UART via the KitProg3 USB-serial bridge (baud rate **115200**). Press the **Reset** (black) button on the board to start.

Choose one of the options below to view the output:

- **Option A** – Open an external terminal application
  - **Tera Term** (Windows)
  - **PuTTY** (Windows)
  - **screen** (Linux / macOS)
- **Option B** – Launch a Python serial reader in a new terminal window (cross-platform)

In [ ]:
from serial_monitor import detect_serial_port, open_tera_term, open_putty, print_screen_cmd, open_serial_terminal

SERIAL_PORT, BAUD_RATE = detect_serial_port()

### Option A – Open an external terminal application

Run one of the cells below to launch a serial terminal pre-configured with the detected port and baud rate. Skip this option if you prefer to read serial output in Python (Option B).

- **Tera Term** – Windows
- **PuTTY** – Windows
- **screen** – Linux / macOS (pre-installed on most systems)

#### Tera Term

In [ ]:
open_tera_term(SERIAL_PORT, BAUD_RATE)

#### PuTTY

In [ ]:
open_putty(SERIAL_PORT, BAUD_RATE)

#### screen

In [ ]:
print_screen_cmd(SERIAL_PORT, BAUD_RATE)

### Option B – Read serial output in a new terminal window (Python)

Opens a **separate terminal window** running a Python serial reader. The output stays outside the notebook so the cell never grows. Works on Windows, Linux, and macOS.

Close the terminal window or press **Ctrl-C** inside it to stop.

In [ ]:
open_serial_terminal(SERIAL_PORT, BAUD_RATE)